# NSSP ESSENCE Data → Load and EDA

This notebook:
1. **Loads** data from the **NSSP ESSENCE** (Electronic Surveillance System for the Early Notification of Community-based Epidemics) API using the **pynssp** Python package.
2. Runs **basic exploratory data analysis** (EDA), similar to the PLACES ZCTA notebook.

**Requirements:** NSSP ESSENCE account (for live API access). Install: `pip install pynssp pandas matplotlib seaborn`.

**Resources:** [pynssp documentation](https://cdcgov.github.io/pynssp/) | [NSSP ESSENCE](https://www.cdc.gov/nssp/essence.html)

In [1]:
# Setup: pynssp and EDA libraries
%pip install pynssp -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# pynssp: create_profile, get_essence_data; fallback simulated data if no credentials
from pynssp import create_profile, get_essence_data
from pynssp.data import load_simulated_ts

print("pynssp and EDA libraries ready.")

Note: you may need to restart the kernel to use updated packages.


/opt/homebrew/Caskroom/miniforge/base/envs/erdos_ds_environment/lib/python3.12/site-packages/pynssp/data.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


pynssp and EDA libraries ready.


## 1. NSSP user profile (credentials)

ESSENCE API requires an NSSP account. Create a profile once (you'll be prompted for username/password), then save it so you can load it next time without re-entering credentials.

In [2]:
# Create or load NSSP profile for ESSENCE API
PROFILE_PATH = "nssp_profile.pkl"

try:
    if os.path.exists(PROFILE_PATH):
        from pynssp.core.credentials import Credentials
        myProfile = Credentials.unpickle(PROFILE_PATH)
        print("Loaded saved NSSP profile.")
    else:
        raise FileNotFoundError("No saved profile")
except Exception:
    myProfile = create_profile()  # prompts for username & password
    myProfile.pickle(PROFILE_PATH)
    print("Created and saved NSSP profile. Re-run this cell next time to load it.")

Created and saved NSSP profile. Re-run this cell next time to load it.


## 2. Load data from ESSENCE (or simulated)

Pull time series data from the ESSENCE API. Replace the URL with your own from ESSENCE (Build a Chart → copy API URL). If the API call fails (e.g. no credentials or network), we fall back to pynssp's simulated time series for EDA.

In [ ]:
# Example ESSENCE Time Series API URL (replace with your own from ESSENCE UI)
# Build a chart in ESSENCE, then use "API" to copy the time series URL
url_ts = (
    "https://essence.syndromicsurveillance.org/nssp_essence/api/timeSeries?"
    "endDate=9Feb2021&medicalGrouping=injury&percentParam=noPercent&"
    "geographySystem=hospitaldhhsregion&datasource=va_hospdreg&detector=probrepswitch&"
    "startDate=11Nov2020&timeResolution=daily&medicalGroupingSystem=essencesyndromes&"
    "userId=455&aqtTarget=TimeSeries"
)

try:
    nssp_df = get_essence_data(url_ts, profile=myProfile)
    print("Loaded data from NSSP ESSENCE API.")
except Exception as e:
    print(f"ESSENCE API call failed ({e}). Using pynssp simulated time series for EDA.")
    nssp_df = load_simulated_ts()

print("Shape:", nssp_df.shape)
nssp_df.head(10)

## 3. Basic EDA

Summary statistics, missing values, and simple visualizations.

In [ ]:
# Summary: shape, dtypes, missing
print("Shape:", nssp_df.shape)
print("\nColumns:", list(nssp_df.columns))
print("\nDtypes:")
print(nssp_df.dtypes)
print("\nMissing values per column:")
print(nssp_df.isna().sum())
print("\nDescribe (numeric):")
nssp_df.describe()

In [ ]:
# Time series plot (if date + count-like column exist)
date_col = None
for c in ["date", "Date", "timeResolution", "dd", "MMM_dd_yyyy"]:
    if c in nssp_df.columns:
        date_col = c
        break
value_col = None
for c in ["dataCount", "count", "cases", "value", "num"]:
    if c in nssp_df.columns:
        value_col = c
        break
if date_col is None:
    value_col = nssp_df.select_dtypes(include=[np.number]).columns[0] if nssp_df.select_dtypes(include=[np.number]).shape[1] else None

if date_col and value_col:
    plot_df = nssp_df.copy()
    plot_df[date_col] = pd.to_datetime(plot_df[date_col], errors="coerce")
    plot_df = plot_df.dropna(subset=[date_col, value_col])
    if len(plot_df) > 0:
        plot_df = plot_df.sort_values(date_col)
        plt.figure(figsize=(10, 4))
        plt.plot(plot_df[date_col], plot_df[value_col], marker=".", markersize=3)
        plt.xlabel(date_col)
        plt.ylabel(value_col)
        plt.title("NSSP ESSENCE time series")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No date/value columns found for time series plot. Columns:", list(nssp_df.columns))

In [ ]:
# Distributions: histograms for numeric columns
num_cols = nssp_df.select_dtypes(include=[np.number]).columns.tolist()
if num_cols:
    n = len(num_cols)
    fig, axes = plt.subplots(1, min(n, 4), figsize=(4 * min(n, 4), 3))
    if n == 1:
        axes = [axes]
    for i, col in enumerate(num_cols[:4]):
        axes[i].hist(nssp_df[col].dropna(), bins=30, edgecolor="black", alpha=0.7)
        axes[i].set_title(col)
    plt.suptitle("Distribution of numeric variables", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns for histograms.")

---

**Next steps:** Use your own ESSENCE API URL (from ESSENCE → Build a Chart → API) to pull different syndromes, geographies, or date ranges. Save the analysis-ready dataframe with `nssp_df.to_csv("nssp_essence_data.csv", index=False)` if needed.